# Segunda Pre-entrega
## Ingesta, radiografía y funciones de transformación

**Proyecto:** identificación de transacciones sospechosas de lavado de dinero con el dataset SAML-D.

El objetivo de este notebook es cargar el archivo, revisar su estructura y datos faltantes, y crear dos transformaciones iniciales. SAML-D es una base de datos ficticias, es decir que sus registros no describen operaciones de clientes reales.


## Bloque 1 Ingesta de datos y primer vistazo

Se instalan e importan las librerías necesarias (Pandas y Gdown) y se carga el CSV en un DataFrame llamado df.


In [2]:
%pip install pandas gdown

Note: you may need to restart the kernel to use updated packages.


In [3]:
import gdown
import pandas as pd

archivo_id = "1uRs7ojeWVmAB0U5fgeUNBCAh4jqLl8O0"
url = f"https://drive.google.com/uc?id={archivo_id}"

gdown.download(url, "SAML-D.csv", quiet=False)

df = pd.read_csv("SAML-D.csv", low_memory=False)
display(df.head())

Downloading...
From (original): https://drive.google.com/uc?id=1uRs7ojeWVmAB0U5fgeUNBCAh4jqLl8O0
From (redirected): https://drive.google.com/uc?id=1uRs7ojeWVmAB0U5fgeUNBCAh4jqLl8O0&confirm=t&uuid=e535dd60-18f9-4846-a914-c542ad81ba54
To: C:\Users\odrio\SAML-D.csv
100%|██████████| 996M/996M [00:16<00:00, 60.1MB/s] 


,Time,Date,Sender_account,Receiver_account,Amount,Payment_currency,Received_currency,Sender_bank_location,Receiver_bank_location,Payment_type,Is_laundering,Laundering_type
0,10:35:19,2022-10-07,8724731955,2769355426,1459.15,UK pounds,UK pounds,UK,UK,Cash Deposit,0,Normal_Cash_Deposits
1,10:35:20,2022-10-07,1491989064,8401255335,6019.64,UK pounds,Dirham,UK,UAE,Cross-border,0,Normal_Fan_Out
2,10:35:20,2022-10-07,287305149,4404767002,14328.44,UK pounds,UK pounds,UK,UK,Cheque,0,Normal_Small_Fan_Out
3,10:35:21,2022-10-07,5376652437,9600420220,11895.00,UK pounds,UK pounds,UK,UK,ACH,0,Normal_Fan_In
4,10:35:21,2022-10-07,9614186178,3803336972,115.25,UK pounds,UK pounds,UK,UK,Cash Deposit,0,Normal_Cash_Deposits


## Bloque 2 Radiografía técnica del dataset

shape muestra las dimensiones exactas; info() permite revisar tipos y valores no nulos; describe() resume las variables numéricas. La celda final calcula indicadores que ayudan a redactar dos observaciones basadas en los resultados observados.


In [10]:
# Dimensiones exactas: (filas, columnas)
print(f"Dimensiones del dataset: {df.shape[0]:,} filas y {df.shape[1]} columnas")

Dimensiones del dataset: 9,504,852 filas y 12 columnas


In [12]:
# Nombres de columnas y Tipos de dato
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9504852 entries, 0 to 9504851
Data columns (total 12 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   Time                    str    
 1   Date                    str    
 2   Sender_account          int64  
 3   Receiver_account        int64  
 4   Amount                  float64
 5   Payment_currency        str    
 6   Received_currency       str    
 7   Sender_bank_location    str    
 8   Receiver_bank_location  str    
 9   Payment_type            str    
 10  Is_laundering           int64  
 11  Laundering_type         str    
dtypes: float64(1), int64(3), str(8)
memory usage: 870.2 MB


In [19]:
# Resumen estadístico de las variables numéricas.
display(df.describe().style.format("{:,.2f}"))


,Sender_account,Receiver_account,Amount,Is_laundering
count,"9,504,852.00","9,504,852.00","9,504,852.00","9,504,852.00"
mean,"5,006,619,409.97","5,006,006,469.65","8,762.97",0.00
std,"2,885,813,892.13","2,884,762,983.27","25,614.95",0.03
min,"9,018.00","9,018.00",3.73,0.00
25%,"2,513,133,245.00","2,513,219,464.00","2,143.69",0.00
50%,"5,001,016,667.00","5,002,572,491.50","6,113.72",0.00
75%,"7,505,051,146.00","7,502,397,427.00","10,458.46",0.00
max,"9,999,986,843.00","9,999,971,095.00","12,618,498.40",1.00


### Hallazgos de la estadística descriptiva
1. **Sender_account** y **Receiver_account** aparecen entre las variables numéricas. Sin embargo, representan identificadores de cuentas, no cantidades; por eso, sus promedios y desviaciones estándar no tienen una interpretación estadística útil.
2. **Is_laundering** está almacenada como número entero, pero sus valores son binarios: 0 indica una transacción no sospechosa y 1 una sospechosa. Su media es cercana a 0, lo que muestra que los casos sospechosos son una proporción muy pequeña del total.


## Bloque 3 Diagnóstico de datos faltantes

Se calcula el número y porcentaje de valores nulos por columna y se ordenan de mayor a menor porcentaje.


In [33]:
# 1. Calculamos la cantidad de nulos y el porcentaje y lo ordenamos de mayor a menor
nulos = df.isna().sum()
porcentaje_nulos = df.isna().mean()*100
porcentaje_nulos = porcentaje_nulos.sort_values(ascending=False)

# 2. Lo convertimos en un DataFrame para que se vea lindo con display()
tabla_nulos = pd.DataFrame(porcentaje_nulos, columns=['Porcentaje de Nulos (%)'])

tabla_nulos = pd.DataFrame({
    "Cantidad de Nulos": nulos,
    "Porcentaje de Nulos (%)": porcentaje_nulos
}).sort_values("Porcentaje de Nulos (%)", ascending=False)

display(tabla_nulos.style.format({"Porcentaje de Nulos (%)": "{:.4f}%"}))


,Cantidad de Nulos,Porcentaje de Nulos (%)
Time,0,0.0000%
Date,0,0.0000%
Sender_account,0,0.0000%
Receiver_account,0,0.0000%
Amount,0,0.0000%
Payment_currency,0,0.0000%
Received_currency,0,0.0000%
Sender_bank_location,0,0.0000%
Receiver_bank_location,0,0.0000%
Payment_type,0,0.0000%


Ninguna columna presenta valores nulos.

## Bloque 4 Funciones personalizadas

La función clasificar_corredor recorre las transacciones y las clasifica según la ubicación de los bancos y las monedas de envío y recepción. Distingue ubicaciones faltantes, operaciones internacionales y operaciones nacionales con monedas iguales o distintas. Esta clasificación es descriptiva y no determina por sí sola que una transacción sea ilícita.
La función lambda clasifica el monto de cada transacción como bajo (menor a 1.000.000), medio (desde 1.000.000 y menor a 5.000.000) o alto (5.000.000 o más). Los límites son rangos definidos para esta exploración con el objetivo de agrupar las transacciones por monto.
Ambas clasificaciones se agregan al DataFrame como las columnas Tipo_corredor y Categoria_monto.


In [51]:
def clasificar_corredor(origen, destino, moneda_envio, moneda_recepcion):
    
    if pd.isna(origen) or pd.isna(destino):
        return "Ubicación faltante"
    elif origen != destino:
        return "Operación Internacional"
    elif pd.isna(moneda_envio) or pd.isna(moneda_recepcion):
        return "Moneda faltante"
    elif moneda_envio != moneda_recepcion:
        return "Operación Nacional, distinta moneda"
    else:
        return "Operación Nacional, misma moneda"


# Columnas que se usarán, en el orden en que se pasan a la función.
columnas = [
    "Sender_bank_location",
    "Receiver_bank_location",
    "Payment_currency",
    "Received_currency"
]

resultados = []

# Recorre cada fila de esas cuatro columnas.
for fila in df[columnas].itertuples(index=False, name=None):
    resultado = clasificar_corredor(*fila)
    resultados.append(resultado)

# Se agrega la clasificación al DataFrame.
df["Tipo_corredor"] = resultados

# Lambda: asigna una categoría según el monto de la transacción.
clasificar_monto = lambda monto: (
    "Monto bajo" if monto < 1000000
    else "Monto medio" if monto < 5000000
    else "Monto alto"
)

df["Categoria_monto"] = df["Amount"].apply(clasificar_monto)

# Se muestran 10 filas en forma aleatoria de las columnas que nos interesan y la columna creada,para comprobar el resultado.
display(df[columnas + ["Tipo_corredor"] + ["Categoria_monto"]].sample(10))


,Sender_bank_location,Receiver_bank_location,Payment_currency,Received_currency,Tipo_corredor,Categoria_monto
8066,UK,UK,UK pounds,UK pounds,"Operación Nacional, misma moneda",Monto bajo
7113556,UK,UK,UK pounds,UK pounds,"Operación Nacional, misma moneda",Monto bajo
6671940,UK,UK,UK pounds,UK pounds,"Operación Nacional, misma moneda",Monto bajo
4599749,UK,UK,UK pounds,UK pounds,"Operación Nacional, misma moneda",Monto bajo
5164513,UK,UK,UK pounds,UK pounds,"Operación Nacional, misma moneda",Monto bajo
5521506,UK,UK,UK pounds,UK pounds,"Operación Nacional, misma moneda",Monto bajo
8826902,UK,UK,UK pounds,UK pounds,"Operación Nacional, misma moneda",Monto bajo
1079911,UK,UK,UK pounds,UK pounds,"Operación Nacional, misma moneda",Monto bajo
5946205,UK,UK,UK pounds,UK pounds,"Operación Nacional, misma moneda",Monto bajo
4034939,UK,UK,UK pounds,UK pounds,"Operación Nacional, misma moneda",Monto bajo


In [50]:
# Revisa cuántas filas quedaron en cada categoría, combinando las dos columnas creadas
display(
    df[["Tipo_corredor"]+ ["Categoria_monto"]].value_counts(dropna=False)
    .rename("cantidad")
    .to_frame()
)


cantidad
Tipo_corredor                       Categoria_monto          
Operación Nacional, misma moneda    Monto bajo        8414768
Operación Internacional             Monto bajo         934342
Operación Nacional, distinta moneda Monto bajo         155688
Operación Nacional, misma moneda    Monto medio            28
                                    Monto alto             16
Operación Internacional             Monto medio             4
Operación Nacional, distinta moneda Monto medio             3
Operación Internacional             Monto alto              2
Operación Nacional, distinta moneda Monto alto              1